In [1]:
import geopandas as gpd


ruta = r"C:\Users\luisc\Documents\Github\analisis-geoespacial\incendios-magnitud-meteorologia\data\raw\incendios\poligonos\if_magnitud_compilado.gpkg"

# Carga el GeoPackage y muestra su tabla de atributos.
capas = gpd.list_layers(ruta)
print("Capas disponibles:")
print(capas)

nombre_capa = capas.iloc[0]["name"]
incendios = gpd.read_file(ruta, layer=nombre_capa)
display(incendios.drop(columns="geometry", errors="ignore"))

Capas disponibles:
                 name   geometry_type
0  incendios_magnitud  MultiPolygon Z


,ID,TEMPORADA,NOM_INCEN,CAUSA,SUPERFICIE,REGION,COMUNA,PROVINCIA,FECHA_INI,FECHA_TER,CAPA_ORIGEN,TEMPORADA_ORIGEN,NUMERO_REG,HUSO,SUP,Superficie_ha
0,1.0,2013-2014,Santa Zenada-deuco,Otros incendios no clasificados,574.000000,RegiÃ³n de La AraucanÃ­a,NaN,NaN,2014-01-05 00:00:00,NaN,if_magnitud_2013_2014,2013_2014,NaN,NaN,NaN,572.938015
1,2.0,2013-2014,Fundo Monaco,Incendio Intencional,207.700000,RegiÃ³n de La AraucanÃ­a,NaN,NaN,2013-12-25 00:00:00,NaN,if_magnitud_2013_2014,2013_2014,NaN,NaN,NaN,207.322936
2,3.0,2013-2014,Toquihue,Incendio Intencional,891.300000,RegiÃ³n de La AraucanÃ­a,NaN,NaN,2014-01-14 00:00:00,NaN,if_magnitud_2013_2014,2013_2014,NaN,NaN,NaN,889.997328
3,28.0,2013-2014,Rumena,Incendios intencionales,3268.900000,RegiÃ³n del BiobÃ­o,NaN,NaN,2013-12-21 00:00:00,NaN,if_magnitud_2013_2014,2013_2014,NaN,NaN,NaN,3258.146300
4,4.0,2013-2014,Miraflores,ElaboraciÃ³n de carbÃ³n,1057.800000,RegiÃ³n de La AraucanÃ­a,NaN,NaN,2014-01-13 00:00:00,NaN,if_magnitud_2013_2014,2013_2014,NaN,NaN,NaN,1055.851552
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
776,NaN,2024-2025,83 - FUNDO EL ESCORIAL,1.2.2. PartÃ­culas incandescentes generadas p...,478.022342,O'Higgins,Marchigue,Cardenal Caro,2024-12-02 00:00:00,2024-12-03 00:00:00,if_magnitud_2024_2025,2024_2025,NaN,NaN,NaN,477.671879
777,NaN,2024-2025,274 - VEGA HONDA,4.2.1. Quema no avisada de desechos agrÃ­colas...,774.982079,Ãuble,San Ignacio,DiguillÃ­n,2025-02-06 00:00:00,2025-04-21 00:00:00,if_magnitud_2024_2025,2024_2025,NaN,NaN,NaN,774.272086
778,NaN,2024-2025,290 - PATAGUAL,4.2.1. Quema no avisada de desechos agrÃ­colas...,219.833516,Ãuble,Pinto,DiguillÃ­n,2025-02-08 00:00:00,2025-02-19 00:00:00,if_magnitud_2024_2025,2024_2025,NaN,NaN,NaN,219.635167
779,NaN,2024-2025,306 - SAN PATRICIO,4.11.3. Empleo de fuentes de calor en faena de...,1938.760474,Ãuble,Coihueco,Punilla,2025-02-14 00:00:00,2025-03-23 00:00:00,if_magnitud_2024_2025,2024_2025,NaN,NaN,NaN,1937.839180


In [2]:
import pandas as pd

# Convertir FECHA_INI y conservar únicamente día, mes y año
incendios["FECHA_INI"] = pd.to_datetime(
    incendios["FECHA_INI"], errors="coerce"
).dt.date

claves = ["NOM_INCEN", "TEMPORADA", "FECHA_INI"]

# Usar "first" para las demás columnas y sumar Superficie_ha
columnas_agregadas = {
    columna: "first"
    for columna in incendios.columns
    if columna not in claves + ["geometry", "Superficie_ha"]
}
columnas_agregadas["Superficie_ha"] = "sum"

# Agrupar incendios repetidos y unir sus geometrías
incendios_agrupados = incendios.dissolve(
    by=claves,
    aggfunc=columnas_agregadas,
    as_index=False,
    dropna=False
)

display(incendios_agrupados.drop(columns="geometry", errors="ignore"))
print(f"Incendios originales: {len(incendios)}")
print(f"Incendios agrupados: {len(incendios_agrupados)}")

,NOM_INCEN,TEMPORADA,FECHA_INI,ID,CAUSA,SUPERFICIE,REGION,COMUNA,PROVINCIA,FECHA_TER,CAPA_ORIGEN,TEMPORADA_ORIGEN,NUMERO_REG,HUSO,SUP,Superficie_ha
0,1-EL ÑANDU,2021-2022,2021-07-02,44.0,1.6.3. Incendio estructural (campamento forest...,1729.330000,Aysen,Cochrane,Capitán Prat,2021-07-07 00:00:00,if_magnitud_2021_2022,2021_2022,NaN,NaN,NaN,1728.652713
1,10 - FRENTE A PAPA VAKA,2024-2025,2024-10-08,NaN,NaN,303.760659,ValparaÃ­so,Isla de Pascua,Isla de Pascua,2025-06-18 00:00:00,if_magnitud_islapascua_2024_2025,islapascua_2024_2025,NaN,NaN,NaN,202.800788
2,1003 - CULLINCO ALTO,2020-2021,2021-01-21,8.0,2.1.2. Conflicto entre personas (venganza con...,239.100000,R.La Araucania,Cholchol,Cautcn,2021-01-25 00:00:00,if_magnitud_2020_2021,2020_2021,NaN,NaN,NaN,238.603203
3,1009 - MEMBRILLAR,2020-2021,2021-01-18,7.0,1.10.6. Otras actividades no clasificados (coc...,679.980000,R.Biobio,Nacimiento,Biobmo,2021-01-29 00:00:00,if_magnitud_2020_2021,2020_2021,NaN,NaN,NaN,678.607160
4,100_ESMERALDA,2016-2017,2016-11-18,126.0,1.7.1. Uso de fuego por transeÃºntes,486.000000,RegiÃ³n Metropolitana de Santiago,Colina,Chacabuco,2016-11-21 00:00:00,if_magnitud_2016_2017,2016_2017,NaN,NaN,NaN,486.178261
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
738,Villucura,2022-2023,NaN,NaN,1.10.6. Otras actividades no clasificados (coc...,1645.399859,Biobío,Santa Barbara,Biobio,6-abr-2023 11:51,if_magnitud_2022_2023,2022_2023,808.0,NaN,NaN,1643.864293
739,ViÃ±a ErrÃ¡zuriz,2014-2015,2014-12-22,57.0,1.4.1. Uso de fuego para actividades recreativ...,819.300000,ValparaÃ­so,Hijuelas,Quillota,NaN,if_magnitud_2014_2015,2014_2015,NaN,NaN,NaN,819.135355
740,VolcÃ¡n Tata Jachura,2023-2024,NaN,22.0,5.1.1. Se investiga pero no es posible estable...,300.894179,TarapacÃ¡,Huara,Tamarugal,16-ago-2023 16:31,if_magnitud_2023_2024,2023_2024,1.0,19K,3.008942e+06,301.133362
741,Ãguila sur,2014-2015,2015-01-10,62.0,1.7.1. Uso de fuego por transeÃºntes,605.900000,Metropolitana,Paine,Maipo,NaN,if_magnitud_2014_2015,2014_2015,NaN,NaN,NaN,605.968428


Incendios originales: 781
Incendios agrupados: 743


In [4]:
import os
import tempfile

directorio = os.path.dirname(ruta)

with tempfile.TemporaryDirectory(dir=directorio) as carpeta_temporal:
    ruta_temporal = os.path.join(carpeta_temporal, "incendios_agrupados.gpkg")

    incendios_agrupados.to_file(
        ruta_temporal,
        layer=nombre_capa,
        driver="GPKG"
    )

    os.replace(ruta_temporal, ruta)

print(f"Archivo reemplazado correctamente en:\n{ruta}")

Archivo reemplazado correctamente en:
C:\Users\luisc\Documents\Github\analisis-geoespacial\incendios-magnitud-meteorologia\data\raw\incendios\poligonos\if_magnitud_compilado.gpkg
